# Task 1 on Colab / Kaggle -- independent evaluation

Scores the deployed model against the external cosmetics collections in `dataset1/` and
`dataset2/`. Trains nothing, takes a few minutes, writes figures 10 and 11.

Run it **after** the combine notebook. It loads `model_resnet_decoupled.pt` from the arm's
checkpoint directory and lifts the `SmallResNet` class straight out of `01_task1_article_type.ipynb`, so both
have to be in the session: upload the data bundle, then drag in the checkpoint zip that
carries `model_resnet_decoupled.pt`.

**Runtime:** GPU preferred, CPU works. **Reads:** `dataset1/`, `dataset2/`, the checkpoints.

See `README.md` in this folder for the file manifest and the session schedule.


## 0. Colab / Kaggle setup

Three cells, none of them part of the assignment: the settings for this session, a dependency
check, and staging the data. They detect the platform themselves, so the same notebook runs on
both. Everything after them is the repository notebook, unchanged.


In [ ]:
# =========================================================================================
# Control panel -- the only cell in this notebook you edit
# =========================================================================================
# Everything below this cell is the assignment notebook as it stands in the repository.
# Nothing here enters RUN_FINGERPRINT, so a checkpoint trained under these settings is
# accepted by a machine running from an ordinary checkout, and the other way round.

# Which arm of the Section 11 external-data experiment this session trains.
#   "supplied"  the ordinary run: supplied rows only.
#   "enriched"  adds the 697 external crops from preprocessed_datasets/task1_dataset1_arms/.
# Every session in one pass must agree. The two arms write to separate checkpoint
# directories and carry different fingerprints, so neither can contaminate the other.
COLAB_ARM = "supplied"

# The runtime's writable working root, and what REPO_ROOT resolves to. "auto" picks
# /kaggle/working on Kaggle, /content on Colab, and the current directory anywhere else.
# Give it a path to override.
PROJECT_ROOT = "auto"

# The data bundle. On Colab, drag it into the session and leave this alone. On Kaggle you add
# it as a Dataset instead and Kaggle extracts it for you, so there is no zip to name -- the
# staging cell finds the extracted copy under /kaggle/input and this setting goes unused.
# Neither platform reads from or writes to Google Drive.
DATA_ZIP = "task1_colab_data.zip"

# Colab only: if the bundle is not already in the session, open a browser file picker instead
# of stopping. Convenient, but the picker is slower than the sidebar and drops large uploads
# more often, and this file is around 590 MB. Prefer the sidebar.
UPLOAD_IF_MISSING = False

# A silent CPU fallback on a GPU job presents as a hang rather than as an error, so the
# notebook refuses to start instead of running a hundred times too slowly.
COLAB_ALLOW_CPU = False    # this job needs a GPU and refuses to start without one.

# --- Runtime report ----------------------------------------------------------------------
import os
import shutil
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()
IN_COLAB = not ON_KAGGLE and ("google.colab" in sys.modules or Path("/content").is_dir())
PLATFORM = "Kaggle" if ON_KAGGLE else "Colab" if IN_COLAB else "local"

if PROJECT_ROOT == "auto":
    PROJECT_ROOT = ("/kaggle/working" if ON_KAGGLE else
                    "/content" if IN_COLAB else str(Path.cwd()))

# The two automatic roots always exist, but a PROJECT_ROOT typed in by hand need not, and
# disk_usage below would then fail on a path nothing has created yet. The staging cell makes
# this directory anyway; doing it here as well costs nothing and moves the failure off a
# reporting line that has no business being the first thing to break.
Path(PROJECT_ROOT).mkdir(parents=True, exist_ok=True)

print(f"Platform         : {PLATFORM}")
print(f"PROJECT_ROOT     : {PROJECT_ROOT}")
print(f"Python           : {sys.version.split()[0]}")
print(f"CPU              : {os.cpu_count()} logical cores")
print(f"Disk free        : {shutil.disk_usage(PROJECT_ROOT).free / 1e9:.1f} GB")
print(f"Arm              : {COLAB_ARM}")
print(f"ALLOW_CPU        : {COLAB_ALLOW_CPU}")

_smi = shutil.which("nvidia-smi")
_gpu = ""
if _smi:
    _gpu = subprocess.run([_smi, "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
print("GPU              :", _gpu or "none visible")

# A second card is worse than useless under Run All: a notebook kernel is one process and
# uses one GPU, so a T4 x2 session bills two cards, trains on one, and that one T4 is slower
# than the single P100 the same menu offers. Said here, at the top, rather than after the
# imports -- by then the session is already committed.
_gpu_count = len([_line for _line in _gpu.splitlines() if _line.strip()])
if _gpu_count > 1:
    print(f"\n  {_gpu_count} GPUs are attached, but Run All uses ONE of them: a notebook kernel\n"
          "  is a single process. Two ways forward, and the first is usually right:\n"
          "    - For an ordinary Run All, switch to the single-GPU accelerator (P100 on\n"
          "      Kaggle). One P100 beats one T4, and you stop paying quota for an idle card.\n"
          "    - To actually use both, do not Run All. Run this instead, in a new cell:\n"
          "        !python scripts/task1_torchrun.py worker_<job>.ipynb\n"
          "      which starts one process per GPU. See 'Using both GPUs' in the README.")

if not _gpu and not COLAB_ALLOW_CPU:
    if ON_KAGGLE:
        print("\n  This notebook needs a GPU. Notebook options -> Accelerator -> GPU P100\n"
              "  (or GPU T4 x2 if you intend to use the torchrun launcher), then Run All again.")
    else:
        print("\n  This notebook needs a GPU. Runtime -> Change runtime type -> T4 GPU,\n"
              "  then Runtime -> Run all again.")
if _gpu and COLAB_ALLOW_CPU:
    print("\n  This job has no GPU path. You are holding a GPU it will not use;\n"
          "  switch this session to a CPU runtime and give the GPU to another job.")


In [ ]:
# =========================================================================================
# Dependencies
# =========================================================================================
# Colab ships every one of these, so the ordinary path installs nothing and this cell is a
# version record rather than a setup step. It matters because the fitted SVM is persisted
# with joblib, and a scikit-learn pickle is only reliably readable by the version that wrote
# it: run hog_svm, hogsearch and the combine step on the same platform, or expect the
# restore to warn and possibly fail. The .pt checkpoints carry no such constraint.

# --- Thread limits, before anything loads a BLAS -----------------------------------------
# OpenMP, MKL and OpenBLAS each read their thread count once, when the shared library is
# first loaded, and ignore every later change to the environment. Section 1.0 below sets
# these and is the authority on them -- but it runs after this cell, and THIS is the cell
# that first imports numpy and torch. Setting them only there would be a no-op that looks
# like it worked, and the run would quietly use the library defaults instead of every core.
# The value matches Section 1.0's exactly, so the later assignment is a harmless restatement.
import os  # noqa: E402
import sys  # noqa: E402

try:
    _cores = len(os.sched_getaffinity(0))
except AttributeError:                 # not on Linux; no affinity mask to read
    _cores = os.cpu_count() or 1

for _variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
                  "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[_variable] = str(_cores)
os.environ["LOKY_MAX_CPU_COUNT"] = str(_cores)
print(f"Thread limits pinned to {_cores} cores before the first BLAS import.")

_required = {"numpy": "numpy", "pandas": "pandas", "torch": "torch",
             "sklearn": "scikit-learn", "skimage": "scikit-image",
             "matplotlib": "matplotlib", "seaborn": "seaborn", "joblib": "joblib"}

_missing = []
for _module, _package in _required.items():
    try:
        __import__(_module)
    except ImportError:
        _missing.append(_package)

if _missing:
    # Needs the runtime to have internet, which is on by default on Colab and off by default
    # on Kaggle.
    print("Installing:", " ".join(_missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
else:
    print("All dependencies already present; nothing installed.")

import numpy  # noqa: E402
import pandas  # noqa: E402
import skimage  # noqa: E402
import sklearn  # noqa: E402
import torch  # noqa: E402

print(f"\nnumpy {numpy.__version__} | pandas {pandas.__version__} | "
      f"scikit-learn {sklearn.__version__} | scikit-image {skimage.__version__}")
print(f"torch {torch.__version__} | CUDA build {torch.version.cuda} | "
      f"CUDA available {torch.cuda.is_available()}")
print("\nRecord these versions alongside the run: uv.lock does not pin them here.")


In [ ]:
# =========================================================================================
# Staging the uploaded data
# =========================================================================================
# The layout this notebook expects, flattened so everything sits directly under
# PROJECT_ROOT rather than inside a repository checkout:
#
#     /content/
#     |- src/preprocessing.py
#     |- src/task1_ddp.py
#     |- preprocessed_datasets/train_manifest.csv
#     |- preprocessed_datasets/task1_dataset1_arms/<version>/     (enriched arm)
#     |- datasets/train/images_train/*.jpg
#     |- datasets/test/styles_prediction.csv, datasets/test/images_test/   (combine)
#     |- dataset1/, dataset2/                                     (enriched arm, evaluation)
#     |- models/, outputs/, predictions/                          (created here)
#
# Everything lives on the runtime's local disk. Nothing is read from or written to Google
# Drive, and nothing here outlives the runtime -- see the warning this cell prints.

PROJECT = Path(PROJECT_ROOT)
PROJECT.mkdir(parents=True, exist_ok=True)

KAGGLE_INPUT = Path("/kaggle/input")


def input_datasets():
    """The read-only dataset directories Kaggle has mounted, outermost first."""
    if not KAGGLE_INPUT.is_dir():
        return []
    return sorted(path for path in KAGGLE_INPUT.iterdir() if path.is_dir())


def extracted_bundle():
    """The bundle's flat tree, if the platform has already unpacked it for us.

    Kaggle unzips an added Dataset itself and mounts the result at /kaggle/input/<slug>/,
    read-only. So on that platform there is often no archive to unpack -- the tree is simply
    already there, on a filesystem this notebook cannot write to. Finding it here is what
    lets the staging step link it into the writable root instead of copying 590 MB into the
    working quota.

    The tree is located by its marker rather than by an assumed shape, because how deeply a
    Dataset nests depends on how it was built -- uploading the zip, a folder, or a folder of
    folders all give different answers.
    """
    for dataset in input_datasets():
        if (dataset / "src" / "preprocessing.py").is_file():
            return dataset
        for marker in sorted(dataset.rglob("src/preprocessing.py")):
            return marker.parent.parent
    return None


def bundle_archive():
    """The bundle still archived, wherever it is: the working root or a mounted Dataset.

    Kaggle usually extracts an uploaded archive, but not always -- a Dataset can carry the
    zip itself, and then /kaggle/input holds a file rather than a tree. Unpacking it into the
    writable root costs a minute and is better than failing.
    """
    explicit = Path(DATA_ZIP) if Path(DATA_ZIP).is_absolute() else (PROJECT / DATA_ZIP)
    if explicit.is_file():
        return explicit
    for dataset in input_datasets():
        for candidate in sorted(dataset.rglob(Path(DATA_ZIP).name)):
            return candidate
    # Any other archive that is plainly not a checkpoint hand-over.
    for dataset in input_datasets():
        for candidate in sorted(dataset.rglob("*.zip")):
            if not candidate.name.startswith("checkpoints_"):
                return candidate
    return None


def describe_inputs():
    """What is actually mounted, for an error message that ends the guessing."""
    lines = []
    for dataset in input_datasets():
        entries = sorted(dataset.iterdir())
        shown = ", ".join(entry.name for entry in entries[:8])
        if len(entries) > 8:
            shown += f", ... (+{len(entries) - 8} more)"
        lines.append(f"            {dataset.name}/  ->  {shown or '<empty>'}")
    return "\n".join(lines) or "            none"


# --- Stage the data, once per session ----------------------------------------------------
if (PROJECT / "src" / "preprocessing.py").is_file():
    print("Data already staged in this session; skipping the unpack.")

elif extracted_bundle() is not None:
    # Kaggle. Symlink rather than copy: /kaggle/input is read-only but perfectly readable, so
    # copying it would spend minutes and most of the working quota to gain nothing. The run
    # only ever writes to models/, outputs/ and predictions/, which are created below as real
    # directories and are not part of this tree.
    _source = extracted_bundle()
    print(f"Found the data already extracted at {_source}")
    _linked, _copied = 0, 0
    for _entry in sorted(_source.iterdir()):
        _target = PROJECT / _entry.name
        if _target.exists() or _target.is_symlink():
            continue
        # Bundles may include banked models. Keep output trees writable: Kaggle
        # inputs are read-only, and training must be able to add checkpoints here.
        if _entry.name in {"models", "outputs", "predictions"}:
            shutil.copytree(_entry, _target)
            _copied += 1
            continue
        try:
            _target.symlink_to(_entry, target_is_directory=_entry.is_dir())
            _linked += 1
        except OSError:
            # Some runtimes refuse symlinks. Copying is slower and spends the working
            # quota, but it is correct, so it is the fallback rather than a failure.
            if _entry.is_dir():
                shutil.copytree(_entry, _target)
            else:
                shutil.copy2(_entry, _target)
            _copied += 1
    print(f"Staged {_linked + _copied} entries into {PROJECT}: "
          f"{_linked} linked, {_copied} copied.")
    if _copied:
        print("  This runtime does not allow symlinks, so the data was copied instead. That "
              "is\n  correct but slower, and it spends the working quota.")

else:
    _zip = bundle_archive()

    if _zip is None and UPLOAD_IF_MISSING and IN_COLAB:
        from google.colab import files
        print(f"{DATA_ZIP} is not in this session. Choose it in the picker below.")
        _uploaded = files.upload()
        _zip = PROJECT / next(iter(_uploaded))

    if _zip is None:
        raise FileNotFoundError(
            "No data found. This notebook needs the bundle built by "
            "`python scripts/make_colab_bundle.py`\n"
            "from a checkout -- it is ~590 MB and contains src/, preprocessed_datasets/, "
            "datasets/,\ndataset1/ and dataset2/ at its top level.\n\n"
            f"  Colab   drag {Path(DATA_ZIP).name} onto {PROJECT} with the sidebar file "
            f"browser.\n"
            f"          Set UPLOAD_IF_MISSING = True for a picker instead, or point DATA_ZIP "
            f"at it.\n\n"
            "  Kaggle  add the bundle as a Dataset (+ Add Input in the right-hand panel).\n"
            "          Upload the ZIP ITSELF when creating the Dataset -- not the datasets/ "
            "folder,\n"
            "          and not the repository. Kaggle extracts it for you.\n\n"
            "          Looked for src/preprocessing.py at any depth, and for a bundle zip, "
            "in each\n"
            "          of these. What is mounted right now:\n\n"
            f"{describe_inputs()}\n\n"
            "See README.md in the task1-collab folder."
        )

    print(f"Unpacking {_zip.name} ({_zip.stat().st_size / 1e6:.0f} MB) into {PROJECT} ...")
    shutil.unpack_archive(str(_zip), str(PROJECT))
    print("Unpacked.")

# A bundle built before the multi-GPU support was added stages cleanly and then fails at the
# import in Section 1.3, an hour later on a Kaggle commit run. Naming it here instead.
if not (PROJECT / "src" / "task1_ddp.py").is_file():
    raise FileNotFoundError(
        f"{PROJECT}/src/task1_ddp.py is missing, so this bundle predates the multi-GPU "
        "support that Section 1.3 imports. "
        "Rebuild it with `python scripts/make_colab_bundle.py` and re-upload, or drop a copy "
        "of src/task1_ddp.py next to src/preprocessing.py if you only need this one session."
    )

for _name in ("models", "outputs", "predictions"):
    (PROJECT / _name).mkdir(parents=True, exist_ok=True)

# --- Import any checkpoints handed over from another session -----------------------------
# Bring a checkpoints_<job>_<arm>.zip from an earlier session into this one -- dragged onto
# PROJECT on Colab, added as a Dataset on Kaggle -- and it is unpacked into this arm's
# checkpoint directory here. That is how the stage-1 backbone reaches the sweep and grid jobs,
# and how every worker's output reaches the combine run. Files are keyed by name, so several
# sessions' zips merge into one directory without collisions.
CHECKPOINT_IMPORT_DIR = PROJECT / "models" / "task1" / "checkpoints" / COLAB_ARM
CHECKPOINT_IMPORT_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle extracts a zip added as a Dataset, so a checkpoint set may arrive there either still
# archived or already unpacked. Both are handled: the archives below, the loose files after.
_search_dirs = [PROJECT] + input_datasets()

_handed_over = [path for directory in _search_dirs
                for path in sorted(directory.glob("checkpoints_*_" + COLAB_ARM + ".zip"))]
for _archive_path in _handed_over:
    shutil.unpack_archive(str(_archive_path), str(CHECKPOINT_IMPORT_DIR))
    print(f"Imported {_archive_path.name}")

# Loose files carry no arm in their own names, so the Dataset has to say. A checkpoint from
# the other arm would be refused at load time by the fingerprint check -- but it would still
# be sitting in this arm's directory, and the export cell zips that directory wholesale, so
# the wrong-arm file would be handed on to the next session. Requiring the evidence up front
# makes that impossible rather than merely unlikely.
_unlabelled = []
for _directory in input_datasets():
    _loose_files = [path for path in sorted(_directory.glob("*"))
                    if path.suffix in {".pt", ".joblib", ".npz"}]
    if not _loose_files:
        continue

    # Evidence, in the order it is trusted: the Dataset's own name, or a marker file sitting
    # beside the checkpoints. An arm named nowhere is not guessed at.
    _names = f"{_directory.name} " + " ".join(path.name for path in _directory.glob("*.txt"))
    _says_this_arm = COLAB_ARM in _names.lower()
    _says_other_arm = any(arm in _names.lower()
                          for arm in ("supplied", "enriched") if arm != COLAB_ARM)

    if not _says_this_arm:
        _unlabelled.append((_directory.name, len(_loose_files), _says_other_arm))
        continue

    for _loose in _loose_files:
        _target = CHECKPOINT_IMPORT_DIR / _loose.name
        if not _target.exists():
            shutil.copy2(_loose, _target)
            print(f"Imported {_loose.name} from {_directory.name}")

for _name, _count, _other in _unlabelled:
    _why = (f"it names the other arm" if _other
            else f"nothing in it names an arm, and {COLAB_ARM!r} is not guessed at")
    print(f"\nSkipped {_count} loose checkpoint(s) in Dataset {_name!r}: {_why}.")
    if not _other:
        print(f"  If they are {COLAB_ARM!r}, rename the Dataset to include {COLAB_ARM!r} and "
              f"re-run,\n  or upload them as checkpoints_<job>_{COLAB_ARM}.zip, which carries "
              "the arm in its name.")

_banked_now = sorted(path.name for path in CHECKPOINT_IMPORT_DIR.iterdir()
                     if path.suffix in {".pt", ".joblib", ".npz"}
                     and not path.name.startswith("epoch_"))
if _banked_now:
    print(f"Checkpoints present for arm {COLAB_ARM!r}: {len(_banked_now)}")
    for _name in _banked_now:
        print("   ", _name)

# A zip for the other arm is a mistake worth naming: its checkpoints carry a different
# fingerprint and would be refused rather than used, after the session had already run.
_other_arm = [path.name for directory in _search_dirs
              for path in sorted(directory.glob("checkpoints_*.zip"))
              if path not in _handed_over]
if _other_arm:
    print("\nIgnored, they belong to the other arm:", ", ".join(_other_arm))


def resolve_external(relative):
    """Locate an external-collection image from the path recorded in the arms CSV.

    `external_train.csv` stores `notebooks/Task1/dataset1/images/<id>.jpg`, because that is
    where the collection sits in a checkout. Rewriting the CSV is not an option: its
    directory name is a content hash of the split and `make_dataset1_arms.py --check`
    verifies it byte for byte. So the path is resolved rather than the file rewritten, and
    both layouts are accepted -- the nested one a checkout has, and the flat one the bundle
    unpacks. REPO_ROOT is read at call time because it is defined below this cell.
    """
    candidate = REPO_ROOT / relative
    if candidate.is_file():
        return str(candidate)
    parts = Path(relative).parts
    for _collection in ("dataset1", "dataset2"):
        if _collection in parts:
            return str(REPO_ROOT.joinpath(*parts[parts.index(_collection):]))
    return str(candidate)


# --- Verify now, rather than an hour into the run ----------------------------------------
REQUIRED = [
    "src/preprocessing.py",
    "preprocessed_datasets/train_manifest.csv",
    "datasets/train/images_train",
    "dataset1/external_cosmetics.csv",
    "dataset1/images",
    "dataset2/external_cosmetics2.csv",
    "dataset2/images",
    "01_task1_article_type.ipynb",
]
if COLAB_ARM == "enriched":
    REQUIRED = REQUIRED + []

_absent = [item for item in REQUIRED if not (PROJECT / item).exists()]
if _absent:
    raise FileNotFoundError(
        "Missing from " + str(PROJECT) + ":\n  " + "\n  ".join(_absent)
        + "\n\nThe bundle is incomplete for this notebook and this arm. Rebuild it with "
          "`python scripts/make_colab_bundle.py` from a checkout that has the data, and see "
          "the file manifest in README.md."
    )

_train_images = len(list((PROJECT / "datasets/train/images_train").glob("*.jpg")))
print(f"\nStaged: {len(REQUIRED)} required paths present | {_train_images:,} training images")
if _train_images < 38000:
    print(f"  WARNING: expected 38,612 training images, found {_train_images:,}. The bundle "
          "looks truncated,\n  and the split -- so the fingerprint -- will not match the other "
          "sessions.")

if ON_KAGGLE:
    print("\n  /kaggle/working becomes this notebook's output when you Save Version, and is\n"
          "  carried between interactive sessions only with Persistence on (Notebook options\n"
          "  -> Persistence). Run the last cell either way: it collects the results into one\n"
          "  zip, which is what the next session expects to be handed.")
else:
    print("\n  Everything this notebook writes lives on the runtime's local disk and is DELETED\n"
          "  when the runtime disconnects. Run the last cell to download the results before\n"
          "  closing the tab, and do not leave a finished session idle.")


## 1. Setup

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score

# Colab edition: the repository is flattened into PROJECT_ROOT, so there is no parent
# directory to walk up to. The original walk is kept as the fallback, which is what
# lets this same file run unchanged inside an ordinary checkout.
REPO_ROOT = Path(PROJECT_ROOT).resolve()
if not (REPO_ROOT / "src" / "preprocessing.py").is_file():
    REPO_ROOT = next(
        (parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (parent / "src" / "preprocessing.py").is_file()),
        None,
    )
if REPO_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the assignment repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import (IMAGE_TARGET_SIZE, load_image_array, load_manifest, make_split)

COMBINE = REPO_ROOT / "01_task1_article_type.ipynb"
CHECKPOINTS = REPO_ROOT / "models/task1/checkpoints" / COLAB_ARM
FIGURES = REPO_ROOT / "outputs/figures"
FIGURES.mkdir(parents=True, exist_ok=True)

DEVICE = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print("Device:", DEVICE)

### 1.1 The architecture, taken from the training notebook rather than restated

Re-typing `SmallResNet` here would create a second definition that can drift from the one the
weights were written by, and the failure would present as a confusing accuracy drop rather than
as an error. The class is therefore lifted out of the combine notebook's model-definition cell
and executed as-is. The trailing probe instance in that cell is cut off, since it exists only for
a parameter count there.

In [ ]:
notebook_cells = ["".join(c["source"])
                  for c in json.loads(COMBINE.read_text(encoding="utf-8"))["cells"]]
architecture = next(s for s in notebook_cells if "class BasicBlock" in s)
architecture = architecture[:architecture.index("set_seed(RANDOM_STATE)")]

namespace = {"nn": nn, "F": F, "torch": torch, "N_CLASSES": 124}
exec(architecture, namespace)
SmallResNet = namespace["SmallResNet"]
print("Architecture loaded from the combine notebook:", SmallResNet.__name__)

### 1.2 The model, and the constants it was trained under

Class order and normalisation come out of the checkpoints themselves. Recomputing either here
would risk evaluating under a different convention from the one the weights expect — the
normalisation constants in particular are fitted on training rows only and must be applied
unchanged, which is exactly the property Section 2.3 of the training notebook persists them for.

In [ ]:
DEPLOYED = "resnet_decoupled"

reference = torch.load(CHECKPOINTS / f"model_{DEPLOYED}.pt",
                       map_location="cpu", weights_only=False)
CLASSES = list(reference["classes"])
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASSES)}
NORM_MEAN = np.asarray(reference["normalisation_mean"], dtype=np.float32)
NORM_STD = np.asarray(reference["normalisation_std"], dtype=np.float32)
FINGERPRINT = reference["fingerprint"]

print(f"Classes: {len(CLASSES)} | run fingerprint: {FINGERPRINT}")
print(f"Deployed checkpoint: model_{DEPLOYED}.pt")


In [ ]:
@torch.no_grad()
def deployed_probabilities(paths, batch_size=256):
    '''Softmax of the deployed checkpoint, averaged with its horizontal mirror.

    This is the pipeline from Section 8, not an approximation of it: same checkpoint, same
    flip TTA, same probability-space averaging of the two views.
    '''
    paths = list(paths)
    blob = torch.load(CHECKPOINTS / f"model_{DEPLOYED}.pt", map_location="cpu",
                      weights_only=False)
    model = SmallResNet()
    model.load_state_dict(blob["state_dict"])
    model = model.to(DEVICE).eval()

    chunks = []
    for start in range(0, len(paths), batch_size):
        batch = np.stack([load_image_array(p, target_size=IMAGE_TARGET_SIZE, scale=False)
                          for p in paths[start:start + batch_size]]).astype(np.float32) / 255.0
        tensor = torch.from_numpy((batch - NORM_MEAN) / NORM_STD)
        tensor = tensor.permute(0, 3, 1, 2).to(DEVICE)
        probabilities = model(tensor).float().softmax(1)
        mirrored = model(tensor.flip(-1)).float().softmax(1)
        chunks.append(((probabilities + mirrored) / 2).cpu().numpy())
    del model
    return np.concatenate(chunks).astype(np.float64)


def report(name, paths, truth):
    '''Score one collection and return its probabilities.'''
    y = np.array([CLASS_TO_INDEX[t] for t in truth])
    probabilities = deployed_probabilities(paths)
    predicted = probabilities.argmax(1)
    labels = sorted(set(y))
    top5 = np.argpartition(probabilities, -5, axis=1)[:, -5:]
    row = {
        "Collection": name,
        "Images": len(y),
        "Classes": len(labels),
        "Top-1": accuracy_score(y, predicted),
        "Top-5": float((top5 == y[:, None]).any(1).mean()),
        "Macro-F1": f1_score(y, predicted, labels=labels, average="macro", zero_division=0),
        "Mean P(true)": float(probabilities[np.arange(len(y)), y].mean()),
        "Mean confidence": float(probabilities.max(1).mean()),
    }
    return row, probabilities, y, predicted

## 2. Provenance and leakage

An independent evaluation is only independent if the data is genuinely unseen. Two claims have
to hold, and both are checked rather than asserted: the external images are not supplied images,
and the labels name classes the model can actually predict.

`docs/EXTERNAL_DATA_AUDIT.md` records the prior screening — 1,200 dataset1 images reviewed in
contact sheets, 42 flagged and removed with approval, leaving 1,158. That audit establishes label
plausibility. It does **not** establish that the collection is unseen, which is what the hash
comparison below is for.

In [ ]:
manifest = load_manifest("articleType")
train_frame, val_frame = make_split(manifest, "articleType",
                                    validation_share=0.20, random_state=42)
print(f"Supplied manifest: {len(manifest):,} rows "
      f"({len(train_frame):,} train / {len(val_frame):,} validation)")

supplied_hashes = {hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in manifest["path"]}

COLLECTIONS = {
    "dataset1": (REPO_ROOT / "dataset1/external_cosmetics.csv",
                 REPO_ROOT / "dataset1/images"),
    "dataset2": (REPO_ROOT / "dataset2/external_cosmetics2.csv",
                 REPO_ROOT / "dataset2/images"),
}

external = {}
rows = []
for tag, (csv_path, image_dir) in COLLECTIONS.items():
    frame = pd.read_csv(csv_path, dtype={"id": str})
    frame["path"] = frame["id"].map(lambda i: str(image_dir / f"{i}.jpg"))
    frame = frame[frame["path"].map(lambda p: Path(p).is_file())].reset_index(drop=True)
    unknown = sorted(set(frame["articleType"]) - set(CLASSES))
    frame = frame[frame["articleType"].isin(CLASSES)].reset_index(drop=True)
    hashes = {hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in frame["path"]}
    rows.append({
        "Collection": tag,
        "Images": len(frame),
        "Classes": frame["articleType"].nunique(),
        "Byte-identical to supplied": len(hashes & supplied_hashes),
        "Labels outside the model's 124": len(unknown),
        "Source": frame["source"].iloc[0],
    })
    external[tag] = frame

provenance = pd.DataFrame(rows)
display(provenance)
assert provenance["Byte-identical to supplied"].eq(0).all(), "External data overlaps supplied data"
print("\nNo external image is byte-identical to a supplied image: the evaluation is independent.")

### 2.1 What the provenance does and does not establish

The hash check above proves these images are **unseen**, which is the property an independent
evaluation actually requires. It does not establish where they came from, and that distinction
is worth stating before any number below is read.

`docs/INDEPENDENT_EVALUATION_DATA.md` records the full position. In short:

| Established | Not established |
|---|---|
| No external image is byte-identical to a supplied one (0 of 1,857) | dataset1's upstream archive, version and licence — recorded as *pending* |
| Every external label is one of the model's 124 classes | dataset2's Roboflow / CC BY 4.0 claim, which is inherited and unverified |
| dataset1 fully screened; 42 flagged images removed, hashes verified | dataset2 label quality — 105 label flags across 148 of 699 images |

Two consequences for how the result should be read.

**dataset1 alone carries the finding.** It is the fully screened collection with no outstanding
label flags, and on its own it produces the headline result across 1,158 images. dataset2
corroborates it over four further classes but is not load-bearing, so its label-quality problems
do not threaten the conclusion.

**Label noise cannot manufacture a zero.** Even taking dataset2's 21.2% flag rate at face value,
four in five of its labels are sound; a model performing anywhere near its in-domain level would
score well above zero on those alone.

What the unresolved provenance genuinely limits is narrower: without the upstream archive this
collection is not a reproducible benchmark another group could regenerate. It is a valid probe of
domain robustness, and the report presents it as exactly that and nothing more.

## 3. What "outside the scope" means here, measured

The `source` field says `external_cosmetics_coco_v1`: these are crops from COCO, a dataset of
objects photographed **in context**. The supplied catalogue is studio product photography on a
white sweep. That is a covariate shift, and it is worth measuring rather than asserting, because
it is the explanation for everything in Section 5.

In [ ]:
def pixel_statistics(paths, limit=400):
    stack = np.stack([load_image_array(p, target_size=IMAGE_TARGET_SIZE, scale=False)
                      for p in list(paths)[:limit]])
    return {"Mean pixel": stack.mean(), "Std pixel": stack.std(),
            "Near-white fraction": float((stack > 240).mean())}


COSMETIC3 = ["Eyeshadow", "Lipstick", "Nail Polish"]
supplied_cosmetics = manifest[manifest["articleType"].isin(COSMETIC3)]

domain = pd.DataFrame([
    {"Collection": "supplied — all classes", **pixel_statistics(manifest["path"])},
    {"Collection": "supplied — the 3 cosmetic classes", **pixel_statistics(supplied_cosmetics["path"])},
    {"Collection": "external dataset1", **pixel_statistics(external["dataset1"]["path"])},
    {"Collection": "external dataset2", **pixel_statistics(external["dataset2"]["path"])},
])
display(domain.style.format({"Mean pixel": "{:.1f}", "Std pixel": "{:.1f}",
                             "Near-white fraction": "{:.3f}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for label, paths, colour in [("supplied catalogue", manifest["path"], "#4C78A8"),
                             ("external (COCO crops)", external["dataset1"]["path"], "#E45756")]:
    stack = np.stack([load_image_array(p, target_size=IMAGE_TARGET_SIZE, scale=False)
                      for p in list(paths)[:400]])
    axes[0].hist(stack.reshape(-1), bins=64, density=True, alpha=0.55, label=label, color=colour)
    axes[1].hist((stack > 240).mean(axis=(1, 2, 3)), bins=40, density=True, alpha=0.55,
                 label=label, color=colour)
axes[0].set_title("Pixel intensity"); axes[0].set_xlabel("value"); axes[0].legend(fontsize=8)
axes[1].set_title("Per-image near-white fraction"); axes[1].set_xlabel("fraction > 240")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("The supplied catalogue is white-background product photography; the external "
             "collection is not", fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES / "10_domain_shift.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. In-domain control

Before asking whether the model transfers, establish that it can do the task at all on these
classes. Without this control a zero on the external set is ambiguous: it could mean the model
never learned cosmetics, rather than that it cannot recognise them out of domain.

Note how thin the supplied evidence is — these three classes are deep in the tail.

In [ ]:
print("Supplied rows for the three cosmetic classes:")
print(supplied_cosmetics["articleType"].value_counts().to_string())

control_rows = []
row, *_ = report("supplied cosmetics — all rows", supplied_cosmetics["path"],
                 supplied_cosmetics["articleType"])
control_rows.append(row)

supplied_val = val_frame[val_frame["articleType"].isin(COSMETIC3)]
row, *_ = report("supplied cosmetics — validation rows only", supplied_val["path"],
                 supplied_val["articleType"])
control_rows.append(row)

control = pd.DataFrame(control_rows)
display(control.style.format({c: "{:.4f}" for c in
                              ["Top-1", "Top-5", "Macro-F1", "Mean P(true)", "Mean confidence"]}))

**The model can do this task.** On supplied imagery of exactly these three classes it reaches
0.97 top-1 across all 39 rows. The validation-only row is weaker and much noisier, which is what
eight images buys you — that thinness is itself part of the finding, and Section 10.2 of the
training notebook already records unmeasured tail classes as a limitation.

## 5. Out-of-domain evaluation

The same three classes, the same model, photographed in context instead of on a white sweep.

In [ ]:
results, probability_store = [], {}
for tag, frame in external.items():
    row, probabilities, y, predicted = report(f"external {tag}", frame["path"],
                                              frame["articleType"])
    results.append(row)
    probability_store[tag] = (probabilities, y, predicted)

external_results = pd.DataFrame(results)
display(external_results.style.format({c: "{:.4f}" for c in
                                       ["Top-1", "Top-5", "Macro-F1", "Mean P(true)",
                                        "Mean confidence"]}))

uniform = 1 / len(CLASSES)
print(f"\nUniform prior over {len(CLASSES)} classes: {uniform:.4f}")
for tag, (probabilities, y, predicted) in probability_store.items():
    true_probability = probabilities[np.arange(len(y)), y]
    print(f"  {tag}: mean P(true class) = {true_probability.mean():.4f} "
          f"({true_probability.mean() / uniform:.2f}x the uniform prior); "
          f"best single image {true_probability.max():.4f}")

In [ ]:
print("What it predicts instead (external dataset1):")
predicted_names = pd.Series([CLASSES[i] for i in probability_store["dataset1"][2]])
display(predicted_names.value_counts().head(10).rename("images").to_frame())
print("Times it predicted any of the three correct classes:",
      int(predicted_names.isin(COSMETIC3).sum()))

### 5.1 Reading the result

**Top-1 is 0.0000 on 1,857 external images, and so is top-5.** Not one image of either collection
has its correct class anywhere in the model's five most likely labels. The mean probability
assigned to the true class is *below* the uniform prior of 1/124 — the model is not merely
uninformed here, it is actively steered away from the right answer.

Two things make this a strong result rather than a broken one:

- **The control rules out the obvious alternative.** The same model scores 0.97 top-1 on the
  same three classes in supplied imagery. It knows what a lipstick looks like on a white sweep.
- **What it predicts instead is coherent.** The errors concentrate on `Handbags`, `Bra`,
  `Briefs`, `Lounge Pants` — large, soft, centrally-framed objects. Against a cluttered natural
  background the model is reading the silhouette of the whole frame, which is precisely the cue
  Section 2.3 of notebook 00 identified as the dominant signal and which HOG exploited as the
  Section 3 baseline. The failure is a direct consequence of the feature the model was
  rewarded for learning.

### 5.2 The part that matters for deployment

Section 8.4 of the training notebook converts calibration into an operating policy: 82.6% of the
catalogue auto-tagged at 95% accuracy, on an ECE of 0.0305. That policy assumes confidence means
what it did in validation.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for tag, colour in [("dataset1", "#E45756"), ("dataset2", "#F58518")]:
    ax.hist(probability_store[tag][0].max(1), bins=40, alpha=0.6, density=True,
            label=f"external {tag} (100% wrong)", color=colour)
ax.axvline(0.5, ls="--", lw=1, color="#333")
ax.set_xlabel("confidence in the predicted class")
ax.set_ylabel("density")
ax.set_title("Confidence on out-of-domain images, every one of which is misclassified",
             fontsize=10)
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES / "11_confidence_under_shift.png", dpi=200, bbox_inches="tight")
plt.show()

for tag, (probabilities, y, predicted) in probability_store.items():
    confidence = probabilities.max(1)
    for threshold in (0.5, 0.8, 0.9):
        share = float((confidence >= threshold).mean())
        print(f"  {tag}: {share:6.1%} of images pass a {threshold:.0%} confidence gate "
              f"— and {share:.1%} of the catalogue would be auto-tagged wrongly")

### 5.2 Can the Failure Be Detected Automatically?

Section 5.1 establishes that the confidence score does not protect anyone: these images are
100% misclassified and a large share of them still clear a 50% confidence gate. Section 8.7 of
the training notebook draws the only safe conclusion available to it -- gate on input
provenance, not on the model's confidence -- which is a manual control and does not scale.

Section 8.6 of that notebook fits an alternative and this section tests it. The gate scores an
image by its squared Mahalanobis distance to the nearest class centroid in the model's own
512-dimensional feature space (Lee et al., NeurIPS 2018), using centroids, a pooled shrunk
covariance and a rejection threshold **all estimated on the training split alone**. Nothing here
refits it and nothing here tunes the threshold: the file is loaded and applied as it was
written, which is the only way this can be a test rather than a demonstration.

Two comparisons make the result readable.

- **Against maximum softmax probability** (Hendrycks & Gimpel, ICLR 2017), the standard
  baseline and the thing already known to fail here. If the distance score does no better, the
  honest finding is that this failure is not detectable from the model's own representation.
- **Against the in-domain validation split**, the same rows and the same split Section 2.1 of
  the training notebook fixed and every model in this project is scored on. That population is
  what "in distribution" means for this system, so it is what the false-rejection rate has to be
  measured against.

A caveat that must travel with any positive result: the two populations differ in source,
framing and capture conditions all at once, so a high AUROC measures "COCO photograph versus
catalogue photograph" and not purely "unfamiliar versus familiar". The in-domain control in
Section 4 is what keeps that honest -- it holds the classes fixed and varies only the imagery.

In [ ]:
from sklearn.metrics import roc_auc_score

GATE_PATH = REPO_ROOT / "models/task1/task1_ood_gate.pt"

if not GATE_PATH.is_file():
    print(f"No gate at {GATE_PATH}.")
    print("Run Section 8.6 of 01_task1_article_type.ipynb first; it writes this file.")
else:
    gate = torch.load(GATE_PATH, map_location="cpu", weights_only=False)
    assert gate["fingerprint"] == FINGERPRINT, (
        f"Gate was fitted under fingerprint {gate['fingerprint']} but this notebook loaded a "
        f"model at {FINGERPRINT}. Refitting one or the other is required; applying a gate "
        f"across a recipe change would measure the change, not the domain shift."
    )
    OOD_MEANS = gate["means"].astype(np.float64)
    OOD_PRECISION = gate["precision"].astype(np.float64)
    OOD_THRESHOLD = gate["threshold"]

    def mahalanobis_scores(features, means=None, precision=None):
        """Squared distance to the nearest centroid. Identical algebra to Section 8.6."""
        means = OOD_MEANS if means is None else means
        precision = OOD_PRECISION if precision is None else precision
        projected = features @ precision
        own = (projected * features).sum(axis=1)
        cross = projected @ means.T
        centroid = ((means @ precision) * means).sum(axis=1)
        return own + (centroid[None, :] - 2.0 * cross).min(axis=1)

    @torch.no_grad()
    def embed_and_score(paths, batch_size=256):
        """Penultimate features and plain-view softmax, from one forward pass per image.

        The gate scores the input, not the prediction, so this deliberately uses the single
        unflipped view rather than the deployed flip-averaged pipeline.
        """
        paths = list(paths)
        blob = torch.load(CHECKPOINTS / f"model_{DEPLOYED}.pt", map_location="cpu",
                          weights_only=False)
        model = SmallResNet()
        model.load_state_dict(blob["state_dict"])
        model = model.to(DEVICE).eval()

        features, probabilities = [], []
        for start in range(0, len(paths), batch_size):
            batch = np.stack([load_image_array(p, target_size=IMAGE_TARGET_SIZE, scale=False)
                              for p in paths[start:start + batch_size]]).astype(np.float32) / 255.0
            tensor = torch.from_numpy((batch - NORM_MEAN) / NORM_STD)
            tensor = tensor.permute(0, 3, 1, 2).to(DEVICE)
            embedded = model.embed(tensor)
            features.append(embedded.float().cpu().numpy().astype(np.float64))
            probabilities.append(model.fc(embedded).float().softmax(1).cpu().numpy())
        del model
        return np.concatenate(features), np.concatenate(probabilities)

    # In-domain reference: the whole validation split, as fixed in Section 2.1 of the
    # training notebook. Same rows every model in this project is scored on.
    id_features, id_probabilities = embed_and_score(val_frame["path"])
    id_distance = mahalanobis_scores(id_features)
    id_confidence = id_probabilities.max(1)

    gate_rows = []
    for tag, frame in external.items():
        ood_features, ood_probabilities = embed_and_score(frame["path"])
        ood_distance = mahalanobis_scores(ood_features)
        ood_confidence = ood_probabilities.max(1)

        # OOD is the positive class, so a detector that works scores it higher. Confidence
        # runs the other way, hence the negation.
        truth = np.r_[np.zeros(len(id_distance)), np.ones(len(ood_distance))]
        auroc_distance = roc_auc_score(truth, np.r_[id_distance, ood_distance])
        auroc_confidence = roc_auc_score(truth, np.r_[-id_confidence, -ood_confidence])

        # FPR@95TPR: with the threshold placed so 95% of in-domain images are accepted, what
        # share of out-of-domain images is accepted too? Lower is better.
        accept_distance = np.quantile(id_distance, 0.95)
        accept_confidence = np.quantile(-id_confidence, 0.95)
        gate_rows.append({
            "External set": tag,
            "AUROC (distance)": auroc_distance,
            "AUROC (confidence)": auroc_confidence,
            "FPR@95 (distance)": float((ood_distance <= accept_distance).mean()),
            "FPR@95 (confidence)": float((-ood_confidence <= accept_confidence).mean()),
            "Rejected at shipped threshold": float((ood_distance > OOD_THRESHOLD).mean()),
        })
        globals()[f"ood_distance_{tag}"] = ood_distance

    gate_table = pd.DataFrame(gate_rows)
    display(gate_table.style.format({c: "{:.4f}" for c in gate_table.columns
                                     if c != "External set"}))

    id_rejected = float((id_distance > OOD_THRESHOLD).mean())
    print(f"In-domain validation rejected at the shipped threshold: {id_rejected:.2%} "
          f"(the gate was set to reject 5% of training rows).")
    print("AUROC 0.5 means the score carries no information about domain; 1.0 means the two "
          "populations are perfectly separable by it.")

    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    ax.hist(np.log10(id_distance), bins=60, alpha=0.65, density=True,
            label=f"in-domain validation (n={len(id_distance):,})", color="#4C78A8")
    for tag, colour in [("dataset1", "#E45756"), ("dataset2", "#F58518")]:
        if f"ood_distance_{tag}" in globals():
            ax.hist(np.log10(globals()[f"ood_distance_{tag}"]), bins=60, alpha=0.6,
                    density=True, label=f"external {tag}", color=colour)
    ax.axvline(np.log10(OOD_THRESHOLD), ls="--", lw=1.2, color="#333",
               label="shipped rejection threshold")
    ax.set_xlabel("log10 squared Mahalanobis distance to the nearest class centroid")
    ax.set_ylabel("density")
    ax.set_title("Does the model's own feature space reveal that an image is foreign?",
                 fontsize=10)
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(FIGURES / "12_ood_gate.png", dpi=200, bbox_inches="tight")
    plt.show()


The confidence signal does not degrade gracefully. A meaningful share of these images clear a
0.5 confidence gate while being **100% wrong**, so the auto-tagging policy would not route them
to human review — it would file them silently. An operating threshold calibrated in domain is
not a safety mechanism out of domain, and the deployment recommendation has to say so.

## 6. Comparison with published work

The second half of the independent-evaluation requirement. Three published efforts on this same
Fashion Product Images dataset, against this project's numbers.

In [ ]:
comparison = pd.DataFrame([
    {"Work": "Condition-CNN (Kolisnik et al., 2021)",
     "Backbone": "VGG16, ImageNet-pretrained",
     "Label space": "articleType as hierarchy level 3",
     "Top-1": "0.910", "Macro-F1": "not reported"},
    {"Work": "uditarora, multitask ResNet50 (GitHub)",
     "Backbone": "ResNet50, ImageNet-pretrained",
     "Label space": "top-20 articleType classes only",
     "Top-1": "0.8835", "Macro-F1": "not reported"},
    {"Work": "Li et al. (2020), arXiv:2005.08170",
     "Backbone": "pretrained CNNs",
     "Label space": "articleType; documents the imbalance",
     "Top-1": "not directly comparable", "Macro-F1": "not reported"},
    {"Work": "This project — ResNet + decoupled classifier, flip TTA",
     "Backbone": "SmallResNet, trained from scratch",
     "Label space": "all 124 articleType classes",
     "Top-1": "0.8774", "Macro-F1": "0.7693"},
])
display(comparison)

### 6.1 Reading the comparison

These are not head-to-head numbers and should not be reported as though they were: each work
uses its own split, and Condition-CNN and the ResNet50 repository both use ImageNet-pretrained
backbones, which this assignment forbids in a submitted model. With that said, three things
survive the caveat.

**The gap to pretrained work is small, and this model starts from noise.** Condition-CNN reports
0.910 top-1 with a pretrained VGG16; this project reaches 0.8774 across the full 124-class label
space with no pretrained weights at all. Roughly three points is a modest price for dropping
ImageNet entirely.

**The most flattering published comparison is the least comparable.** The ResNet50 repository's
0.8835 is measured over the *twenty most frequent* classes. This project's 0.8774 spans all 124,
including the 40 classes with fewer than 30 training images. Restricting to a head-only label
space removes exactly the part of the problem that is hard.

**None of them report macro-F1, and on this data that is the whole argument.** At 6,584:1
imbalance, accuracy cannot distinguish a model that handles the tail from one that ignores it —
this notebook's own majority-class baseline reaches 0.174 accuracy at 0.0027 macro-F1. A
published top-1 of 0.910 is consistent with a wide range of tail behaviour, none of which is
reported. That is a genuine gap in the comparison and it cuts both ways: it means this project
cannot claim to beat them on the metric it considers most important, because they never
published it.

## 7. What this contributes to the Ultimate Judgement

Three findings, carried into Section 8.7 of the training notebook.

1. **Within its domain the recommendation stands.** 0.97 top-1 on supplied imagery of the three
   external classes, consistent with the in-domain headline.
2. **Outside its domain the model fails completely and silently.** 0/1,857 top-1 *and* top-5,
   with mean true-class probability below the uniform prior, while remaining ~35% confident. The
   deployable claim is bounded to catalogue-style product photography on a plain background.
3. **Against published work it is competitive without pretrained weights**, and it reports the
   tail metric that the published work does not.

The honest conclusion is narrower than the headline metric alone would support, which is the
point of evaluating independently.

In [ ]:
# =========================================================================================
# Collect the figures -- RUN THIS BEFORE CLOSING THE TAB
# =========================================================================================
import zipfile

_zip_path = PROJECT / "task1_independent_evaluation.zip"
_figures = sorted(FIGURES.glob("1[01]_*.png"))

with zipfile.ZipFile(_zip_path, "w", zipfile.ZIP_DEFLATED) as _archive:
    for _path in _figures:
        _archive.write(_path, _path.name)

print(f"Wrote {_zip_path}")
for _path in _figures:
    print(f"  {_path.name}")

if IN_COLAB:
    from google.colab import files
    files.download(str(_zip_path))
elif ON_KAGGLE:
    print(f"\nOn Kaggle: {_zip_path.name} is in /kaggle/working. Download it from the Data\n"
          "panel on the right while this session is alive, or Save Version and take it from\n"
          "the run's Output. To hand it to another Kaggle session, publish that output as a\n"
          "Dataset and add it there with + Add Input.")
else:
    print(f"\nCollect {_zip_path} yourself.")
